In [1]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, END

load_dotenv("../.env")
api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [2]:
### 1단계: State(바구니) - 노드들이 서로 주고받을 데이터 형식
class RobotState(TypedDict):
    question: str
    category: str
    answer: str

In [3]:
### 2단계: Node(일꾼) - 실제 작업을 하는 함수들
def analyzer_node(state: RobotState):
    """질문이 법률인지 아닌지 분석하는 일꾼"""
    print("--- 분석 중 ---")
    question = state["question"]
    # 간단하게 구분하는 로직 (실제로는 LLM을 써야 함)
    category = "legal" if "법" in question or "민법" in question else "general"
    return {"category": category}

def generator_node(state: RobotState):
    """답변을 생성하는 일꾼"""
    print("--- 답변 생성 중 ---")
    res = llm.invoke(f"{state['question']}에 대해 짧게 대답해줘.")
    return {"answer": res.content}

def reject_node(state: RobotState):
    """거절 메시지를 만드는 일꾼"""
    print("--- 거절 처리 중 ---")
    return {"answer": "죄송합니다. 법률 관련 질문만 답변 가능합니다."}

In [4]:
### 3단계: 그래프 조립 (설계도)
workflow = StateGraph(RobotState)

# 노드 등록
workflow.add_node("analyzer", analyzer_node)
workflow.add_node("generator", generator_node)
workflow.add_node("rejector", reject_node)

# 길(Edge) 연결
workflow.set_entry_point("analyzer") # 시작점은 분석 노드

# 조건부 길찾기 (Conditional Edge)
def decide_next_node(state: RobotState):
    if state["category"] == "legal":
        return "generator" # 법률이면 답변 생성으로
    return "rejector"     # 아니면 거절 노드로

workflow.add_conditional_edges(
    "analyzer",           # 분석 노드가 끝나면
    decide_next_node      # 이 함수가 결정한 곳으로 가라
)

# 마무리 연결
workflow.add_edge("generator", END)
workflow.add_edge("rejector", END)

In [5]:
### 4단계: 실행
app = workflow.compile()

# 테스트 1: 법률 질문
print("\n[테스트 1: 민법 질문]")
final_state = app.invoke({"question": "민법 제1조가 뭐야?"})
print(f"최종 답변: {final_state['answer']}")

# 테스트 2: 일상 질문
print("\n[테스트 2: 점심 메뉴 질문]")
final_state = app.invoke({"question": "오늘 점심 뭐 먹을까?"})
print(f"최종 답변: {final_state['answer']}")


[테스트 1: 민법 질문]
--- 분석 중 ---
--- 답변 생성 중 ---
최종 답변: 민법 제1조는 법원이 민사 사건을 판단할 때 적용해야 할 **법의 근거(法源)**를 정해놓은 조항입니다.

즉, **법률**에 규정이 없으면 **관습법**에 따르고, 관습법도 없으면 **조리**에 따른다고 규정하고 있어요.

[테스트 2: 점심 메뉴 질문]
--- 분석 중 ---
--- 거절 처리 중 ---
최종 답변: 죄송합니다. 법률 관련 질문만 답변 가능합니다.
